In [12]:
# https://colab.research.google.com/drive/1JMLa53HDuA-i7ZBmqV7ZnA3c_fvtXnx-?usp=sharing#scrollTo=wJpXpmjEYC_T
# https://www.bilibili.com/video/BV1BbFaeVE4W  PyTorch手搓Transformer
# https://github.com/hankinghu/literature-books/tree/master

In [61]:

import torch
import torch.nn as nn
from torch.nn import functional as F
import textwrap
import random

# 超参数
file_name="sanguo-all.txt"
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
wrap_width = 50
max_iters = 6000
eval_interval = 500
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.1
# ------------

torch.manual_seed(1337)


In [62]:
# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open(file_name, 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


In [63]:
# Head类
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__ ()
        self.value = nn.Linear(n_embd,head_size,bias=False)# 线性变换层
        self.register_buffer("tril",torch.tril(torch.ones(block_size,block_size)))#不可训练的,结构(约等于常量)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        wei = torch.ones(T,T,device=device)  #注意力方阵(B，T，T)
        wei = wei.masked_fill(self.tril == 0,float("-inf"))# 掩码填充
        wei =F.softmax(wei,dim=-1)
        wei = self.dropout(wei)  # 随机去掉(归零)一些值，增加网络的稳定性
        v = self.value(x)
        out = wei @ v   #(B, T, head size)
        return out

In [64]:
# 语言模型
class LanguageModel(nn.Module):
    def __init__ (self):
        super().__init__ ()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.head = Head(n_embd)
        self.network = nn.Sequential(
            nn.Linear(n_embd,200),
            nn.ReLU(),
            nn.Linear(200,vocab_size)
        )
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

    def forward(self,idx,targets=None):
        B,T=idx.shape     #(B,T)B= batch_size,T= block_size，数据为token(整数)形式
        token_embd=self.token_embedding_table(idx)
        position_idx= torch.arange(T,device=device)
        position_embd = self.position_embedding_table(position_idx)
        x=token_embd + position_embd #(B,T,n embd)
        head_out = self.head(x)  #添加注意力头
        logits =self.network(head_out)  #(B，T，vocab size)
        if targets is None:
            loss = None
        else:
            B, T, C= logits.shape
            logits =logits.view(B*T, C)  #摊平
            targets = targets.view(B*T)
            loss=F.cross_entropy(logits, targets)

        # B,T=idx.shape #B= batch size,T= block size，数据为token(整数)形式
        # random_tensor = torch.rand(B,T,vocab_size,device=device) #
        # logits = random_tensor /random_tensor.sum(dim=-1, keepdim=True)
        # loss = None
        return logits, loss
    
    def generate(self, token_sequ, max_new_tokens):
        # token_sequ已知的上文,max_new_tokens是续写的长度(B，T)
        for _ in range(max_new_tokens):
            tokens_input = token_sequ[:, -block_size: ]
            logits, loss = self.forward(tokens_input)  # logits,(B, T, vocab size)
            logits = logits[:,-1,:] #只取字符串最后一个,(概率分布向量格式)
            probs =F.softmax(logits,dim=-1)
            token_next = torch.multinomial(probs,num_samples=1)# 概率分布向量-->one-hot 向量-->整数token
            token_sequ =torch.cat((token_sequ, token_next), dim=1)
        new_tokens =token_sequ[:,-max_new_tokens:]
        return new_tokens

In [65]:
#--损失评测--------
@torch.no_grad()   #不做梯度计算的decorator,作用域为整个函数
def estimate_loss(model):
    out = {}
    model.eval()  #把模型转化为evaluate模式(默认模式是train)
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)  # 建立一个初始值为0的容器,用于储存loss值
        for k in range(eval_iters):
            X, Y = get_batch(split)  # split是一个字符串,用来控制get_batch()函数的行为
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()  # out是含有两个元素的字典，一个是train，一个是val，每个元素对应一个loss的平均值
    model.train() # 再转化为训练模式(如果之前没有转为evaluate模式,则不需要这一步,因为模型建立后默认为训练模式)
    return out

In [66]:
def main():
    print(f"训练内容:{file_name}")
    model =LanguageModel()#实例化
    model = model.to(device)
    print(sum(p.numel()for p in model.parameters())/1e6,'M parameters')# 打印有多少个参数
    #设定一个优化器
    optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)
    # 训练循环
    for i in range(max_iters):
        if i % eval_interval ==0 or i==max_iters - 1:
            losses =estimate_loss(model)
            print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        #取样
        xb,yb = get_batch("train")
        logits, loss=model(xb, yb)   #前馈运算
        optimizer.zero_grad()   #把旧的梯度归零
        loss.backward()   #反向传播,计算新的梯度
        optimizer.step()  #做一步优化计算

    print("训练结束，下面开始生成内容")
    max_new_tokens =200
    start_idx = random.randint(0, len(val_data)-block_size-max_new_tokens)
    #上文内容
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)# (B, T)B = 1,T = block size
    context[0,:]=val_data[start_idx:start_idx+block_size]
    context_str =decode(context[0].tolist())#一阶张量
    wrapped_context_str = textwrap.fill(context_str, width=wrap_width)
    #真实下文
    real_next_tokens = torch.zeros((1,max_new_tokens), dtype=torch.long, device=device)
    real_next_tokens[0, :]= val_data[start_idx+block_size: start_idx+block_size+max_new_tokens]
    real_next_tokens_str = decode(real_next_tokens[0].tolist())# 一阶张量
    wrapped_real_next_tokens_str = textwrap.fill(real_next_tokens_str, width=wrap_width)
    #生成下文
    generated_tokens = model.generate(context, max_new_tokens)
    generated_str =decode(generated_tokens[0].tolist())
    wrapped_generated_str = textwrap.fill(generated_str, width=wrap_width)

    print("---------上文内容---------:")
    print(wrapped_context_str)
    print("---------真实上文内容---------:")
    print(wrapped_real_next_tokens_str)
    print("---------生成内容---------:")
    print(wrapped_generated_str)


main()

训练内容:sanguo-all.txt
1.069604 M parameters
step 0: train loss 8.2980, val loss 8.2970
step 500: train loss 6.0845, val loss 6.2917
step 1000: train loss 5.9791, val loss 6.2177
step 1500: train loss 5.8897, val loss 6.1335
step 2000: train loss 5.8095, val loss 6.1122
step 2500: train loss 5.7593, val loss 6.0726
step 3000: train loss 5.7016, val loss 6.0526
step 3500: train loss 5.6902, val loss 6.0271
step 4000: train loss 5.6513, val loss 6.0138
step 4500: train loss 5.6095, val loss 5.9980
step 5000: train loss 5.5896, val loss 5.9793
step 5500: train loss 5.5724, val loss 5.9780
step 5999: train loss 5.5295, val loss 5.9799
训练结束，下面开始生成内容
---------上文内容---------:
毋丘俭听知东兴兵败，亦勒兵而退。 却说诸葛恪引兵至东兴，收兵赏劳
---------真实上文内容---------:
了毕，乃聚诸将曰：“司马昭兵败北归，正好乘势进取中原。”遂一面遣人赍书入蜀，求姜维进兵攻其北，许以平
分天下；一面起大兵二十万，来伐中原。临行时，忽见一道白气，从地而起，遮断三军，对面不见。蒋延曰：“此
气乃白虹也，主丧兵之兆。太傅只可回朝，不可伐魏。”恪大怒曰：“汝安敢出不利之言，以慢吾军心！”叱武士
斩之。众皆告免，恪乃贬蒋延为庶人，仍催兵前进。丁奉曰：“魏以新城为总隘口，若先取得此城，司马师破胆矣
---------生成内容---------:
乃约被四拾里而正一十，近进拨十哨寨循，往西皖。数寨忽围，夜司走魏中，大吕维忽前